In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words)

32033

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [13]:
block_size = 3 # its the sliding window length, model can only see the last three characters to predict the next one
X, Y = [], [] # X stores the inputs, Y stores the targets
for w in words[:5]:
    context = [0] * block_size # thus over here, context is just the list of the last three letters we've seen so far, its initialised to zero, to portray that we've seen nothing
    for ch in w + '.' :
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

... ---> y
..y ---> u
.yu ---> h
yuh ---> e
uhe ---> n
hen ---> g
eng ---> .
... ---> d
..d ---> i
.di ---> o
dio ---> n
ion ---> d
ond ---> r
ndr ---> e
dre ---> .
... ---> x
..x ---> a
.xa ---> v
xav ---> i
avi ---> e
vie ---> n
ien ---> .
... ---> j
..j ---> o
.jo ---> r
jor ---> i
ori ---> .
... ---> j
..j ---> u
.ju ---> a
jua ---> n
uan ---> l
anl ---> u
nlu ---> i
lui ---> s
uis ---> .


In [14]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([36, 3]), torch.int64, torch.Size([36]), torch.int64)

In [15]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words)) # 80% of total words
n2 = int(0.9*len(words)) # 90% of total words

# Xtr = Input tensors for training the context, Ytr = Output tensors for training the next character

Xtr, Ytr = build_dataset(words[:n1])        # First 80%
Xdev, Ydev = build_dataset(words[n1:n2])    # Next 10% (from 80% to 90%)
Xte, Yte = build_dataset(words[n2:])        # Last 10% (from 90% to 100%)

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [16]:
C = torch.randn((27, 2)) # 27 = number of possible characters, 2 = embedding dimension

In [17]:
emb = C[X]
emb.shape

torch.Size([36, 3, 2])

In [18]:
W1 = torch.randn((6, 100)) # Weight matrix(First hidden layer) : 6 inputs --> 100 hidden neurons
b1 = torch.randn(100) # Bias : 100 numbers added to each hidden neuron

In [19]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [20]:
h

tensor([[ 0.6354, -0.9926, -0.9996,  ..., -0.9472,  0.7701, -0.9958],
        [ 0.9995,  0.9987, -0.9972,  ..., -0.9977, -0.3216, -0.9954],
        [ 0.9893,  0.9948, -0.9980,  ..., -0.9477, -1.0000, -0.7327],
        ...,
        [ 0.9330,  0.6253, -0.9982,  ...,  0.0916, -1.0000,  0.2596],
        [ 0.9903,  0.9824,  0.9999,  ..., -1.0000,  0.9990,  0.9998],
        [-0.7105, -0.3888, -0.9976,  ...,  0.8675, -0.8067, -0.9965]])

In [21]:
h.shape

torch.Size([36, 100])

In [31]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [33]:
logits = h @ W2 + b2

In [34]:
logits.shape

torch.Size([36, 27])

In [36]:
counts = logits.exp()

In [37]:
prob = counts / counts.sum(1, keepdims=True)

In [42]:
loss = -prob[torch.arange(prob.shape[0]), Y].log().mean()